# Lesson 22 - Instructor solutions

Attempt the learner notebook first. These are one set of possible solutions, with reasoning and
executable checks. Restart and run this notebook from the top: the worked examples below supply
the same definitions as the learner lesson. No learner answers or state are required.


## 1. From Python records to numerical arrays

Lists can hold mixed objects. A NumPy array stores a rectangular block with one data type
(`dtype`), which makes numerical operations compact and usually faster for sizable numerical data.
Vectorization asks NumPy to perform a whole-array operation in compiled routines; it does not
guarantee every tiny calculation is faster. Keep dictionaries for named records and arrays for measurements.

`ndim` counts axes, `shape` gives lengths along them, and `size` counts all elements.
Rows below are sites; columns are three daily kWh readings. Predict the shape before running.


In [ ]:
import numpy as np

readings = np.array([[10, 12, 14], [20, 22, 24]], dtype=np.float64)
print(readings)
print(readings.ndim, readings.shape, readings.size, readings.dtype)
assert readings.shape == (2, 3)
assert readings.dtype == np.float64


## 2. Create arrays and choose a dtype

`zeros` and `ones` create initialized arrays. `arange` resembles range with an excluded stop;
`linspace` requests a fixed count and includes both endpoints by default. Prefer linspace to
floating steps when endpoint/count matters. A seeded generator provides repeatable synthetic data.
Integer arrays cannot preserve fractional assignments. `astype` explicitly converts the dtype.


In [ ]:
print(np.zeros((2, 2)))
print(np.ones(3, dtype=np.int64))
print(np.arange(0, 6, 2))
print(np.linspace(0, 1, 5))
rng = np.random.default_rng(22)
print(rng.integers(1, 5, size=4))
counts = np.array([1, 2], dtype=np.int64)
counts[0] = 1.9  # Fraction is truncated; this is not rounding.
print(counts, counts.astype(float))
assert counts.tolist() == [1, 2]


## 3. Index, slice, and copy deliberately

`array[row, column]` accesses one value. Slices exclude their stop as in lists, but basic
NumPy slices usually share the original storage: changing a slice changes the source. Use `.copy()`
for an independent result. Boolean/advanced indexing instead creates a copy.
Predict whether the following original changes. Change the copy and compare.


In [ ]:
sample = np.array([[10., 12., 14.], [20., 22., 24.]])
print(sample[1, 2], sample[:, 1], sample[0, :2])
view = sample[:, :2]
view[0, 0] = 99
independent = sample.copy()
independent[0, 0] = -1
print(sample[0, 0], independent[0, 0])
assert sample[0, 0] == 99


## 4. Boolean masks select matching measurements

A comparison creates one boolean per element. Indexing with that mask returns matching values.
Combine array conditions with `&`, `|`, and `~`, wrapping each comparison in parentheses.
Python `and` expects a single truth value and cannot combine whole masks.
`np.where(condition, yes, no)` chooses element by element without changing the source.


In [ ]:
mask = (readings >= 12) & (readings < 23)
selected = readings[mask]
print(mask, selected, sep="\n")
print(np.where(readings > 20, "review", "normal"))
assert selected.tolist() == [12., 14., 20., 22.]


## 5. Reshape and understand axes

Reshaping rearranges the interpretation of existing values while keeping the element count.
One `-1` lets NumPy infer a dimension. A reshape may share memory; use copy when independence matters.
For a `(sites, days)` table, `axis=0` collapses sites and leaves one value per day;
`axis=1` collapses days and leaves one per site. Transpose (`.T`) exchanges the two axes.


In [ ]:
six = np.arange(6)
print(six.reshape(2, 3))
print(readings.T.shape)
print("Per day:", readings.sum(axis=0))
print("Per site:", readings.sum(axis=1))
print("Mean/min/max:", readings.mean(), readings.min(), readings.max())
assert readings.sum(axis=0).tolist() == [30, 34, 38]
assert readings.sum(axis=1).tolist() == [36, 66]


## 6. Vectorization and broadcasting

For lists, `* 2` repeats items; for arrays it multiplies values. `*` is elementwise;
matrix multiplication uses `@`. Broadcasting aligns shapes from the right: paired dimensions
must be equal or one of them must be 1. A scalar broadcasts everywhere; a `(3,)` tariff aligns
with three columns. Per-site factors need shape `(2, 1)`, not `(2,)`.
Predict the first site's bill; change only the final day's tariff and explain which totals move.


In [ ]:
print([10, 12] * 2, np.array([10, 12]) * 2)
tariff = np.array([2., 3., 4.])  # Currency units per kWh, one per day.
cost = readings * tariff
print(cost, cost.sum(axis=1), sep="\n")
site_factors = np.array([1., 0.5]).reshape(-1, 1)
print(readings * site_factors)
assert cost.sum(axis=1).tolist() == [112, 202]


## 7. Conversion boundaries and debugging

Mixed text/numbers can turn the entire array into strings. Ragged nested lists do not form
a numerical rectangle. Diagnose errors by printing shape and dtype before doing arithmetic.
`tolist()` produces ordinary nested Python values for JSON. Check finite values with `isfinite`;
NaN is not equal to itself. Empty arrays need an explicit aggregation policy.
Energy examples use floats; retain Lesson 12's Decimal policy for exact accounting.


In [ ]:
mixed = np.array([10, "12"])
print(mixed.dtype, mixed.astype(float) + 1)
for operation in [lambda: np.array([[1, 2], [3]]),
                  lambda: np.arange(5).reshape(2, 3),
                  lambda: readings + np.ones(2)]:
    try:
        operation()
    except ValueError:
        print("Rejected: inspect rectangularity, element count, or broadcast shapes.")
dirty = np.array([10., np.nan, np.inf])
print(dirty[np.isfinite(dirty)])
assert readings.tolist() == [[10., 12., 14.], [20., 22., 24.]]


## Exercise 1

Choose the dtype at the boundary so later fractional operations have the intended representation.


In [ ]:
practice_a = np.array([5, 10, 15], dtype=np.float64)
assert practice_a.shape == (3,)
assert practice_a.dtype == np.float64
assert practice_a.sum() == 30


## Exercise 2

Use a colon for every row and one column index; a single row index keeps that row.


In [ ]:
practice_grid = np.array([[2,4,6],[8,10,12]])
assert practice_grid[:, 1].tolist() == [4,10]
assert practice_grid[-1].tolist() == [8,10,12]


## Exercise 3

A nonnegative comparison alone still admits infinity. Combine domain and finite checks.


In [ ]:
practice_dirty = np.array([10.,-1.,np.nan,20.,np.inf,0.])
practice_valid = practice_dirty[np.isfinite(practice_dirty) & (practice_dirty >= 0)]
assert practice_valid.tolist() == [10,20,0]
assert practice_valid.mean() == 10


## Exercise 4

The trailing dimension represents days in both arrays; axis 1 sums those days.


In [ ]:
practice_totals = (np.array([[1,2,3],[4,5,6]]) * np.array([2,3,4])).sum(axis=1)
assert practice_totals.tolist() == [20,47]


## Exercise 5

A `(rows, 1)` mean broadcasts across columns. Validation gives a clearer error than an accidental broadcasting failure.


In [ ]:
def practice_center_rows(values):
    a = np.asarray(values, dtype=float)
    if a.ndim != 2 or 0 in a.shape or not np.isfinite(a).all():
        raise ValueError("Expected nonempty finite 2D values")
    return a - a.mean(axis=1, keepdims=True)
practice_source = np.array([[1.,3.],[10.,14.]])
np.testing.assert_allclose(practice_center_rows(practice_source), [[-1,1],[-2,2]])
assert practice_source.tolist() == [[1,3],[10,14]]
for bad in [[], [1,2], [[np.nan]]]:
    try:
        practice_center_rows(bad)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid shape/value accepted")


## Chapter project

Build `energy_report(values, tariffs)` for a nonempty finite nonnegative 2D kWh table and a
finite nonnegative 1D tariff of matching column length. Return JSON-ready `site_kwh`, `site_cost`,
and `grand_cost`. Validate shapes before multiplying; never mutate inputs.

**Check:** `[[10,12,14],[20,22,24]]` and `[2,3,4]` give site kWh `[36,66]`,
cost `[112,202]`, grand cost 314. Reject a short tariff and a negative reading.

The implementation separates inputs, transformation, and reporting.


In [ ]:
def energy_report(values, tariffs):
    a, t = np.asarray(values, dtype=float), np.asarray(tariffs, dtype=float)
    if a.ndim != 2 or 0 in a.shape or t.ndim != 1 or t.size != a.shape[1]:
        raise ValueError("Expected sites x days and one tariff per day")
    if not (np.isfinite(a).all() and np.isfinite(t).all()) or (a < 0).any() or (t < 0).any():
        raise ValueError("Expected finite nonnegative readings and tariffs")
    costs = (a * t).sum(axis=1)
    return {"site_kwh": a.sum(axis=1).tolist(), "site_cost": costs.tolist(),
            "grand_cost": float(costs.sum())}
assert energy_report([[10,12,14],[20,22,24]], [2,3,4]) == {
    "site_kwh": [36,66], "site_cost": [112,202], "grand_cost": 314}
for values, rates in [([[1,2]], [1]), ([[-1]], [2])]:
    try:
        energy_report(values, rates)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid input accepted")
